# Phase 3 – Model Training (SVM & Decision Tree)

Train and tune SVM and Decision Tree classifiers using the pre-extracted image features.

## Section 1: Load & clean dataset
Load feature data (with `label`) generated in Phase 2.

In [2]:
from pathlib import Path
import logging
import pandas as pd
import numpy as np
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, HalvingRandomSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    force=True,
 )

data_path = Path("feature2.csv")
if not data_path.exists():
    fallback_path = Path("feature2.csv")
    if fallback_path.exists():
        logging.info("feature.csv not found. Falling back to feature.csv.")
        data_path = fallback_path
    else:
        raise FileNotFoundError("feature.csv not found (also checked feature.csv).")

logging.info("Loading dataset from %s", data_path)
df = pd.read_csv(data_path)
logging.info("Loaded dataset shape: %s", df.shape)
logging.info("Columns: %s", df.columns.tolist())

df.head()

2025-12-10 08:34:20,671 [INFO] Loading dataset from feature2.csv
2025-12-10 08:34:30,717 [INFO] Loaded dataset shape: (26179, 1869)
2025-12-10 08:34:30,718 [INFO] Columns: ['filename', 'class', 'label', 'aspect_ratio', 'mean_red', 'mean_green', 'mean_blue', 'std_red', 'std_green', 'std_blue', 'mean_hue', 'mean_saturation', 'mean_value', 'std_hue', 'std_saturation', 'std_value', 'brightness', 'contrast_simple', 'mean_intensity', 'std_intensity', 'min_intensity', 'max_intensity', 'range_intensity', 'skewness', 'kurtosis', 'entropy', 'contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM', 'edge_mean', 'edge_std', 'edge_density', 'contour_area', 'contour_perimeter', 'compactness', 'lbp_hist_0', 'lbp_hist_1', 'lbp_hist_2', 'lbp_hist_3', 'lbp_hist_4', 'lbp_hist_5', 'lbp_hist_6', 'lbp_hist_7', 'lbp_hist_8', 'lbp_hist_9', 'lbp_hist_10', 'lbp_hist_11', 'lbp_hist_12', 'lbp_hist_13', 'lbp_hist_14', 'lbp_hist_15', 'lbp_hist_16', 'lbp_hist_17', 'color_hist_b_0', 'color_hist_b_1'

,filename,class,label,aspect_ratio,mean_red,mean_green,mean_blue,std_red,std_green,std_blue,...,hog_1755,hog_1756,hog_1757,hog_1758,hog_1759,hog_1760,hog_1761,hog_1762,hog_1763,split
0,butterfly (1).jpeg,butterfly,0,1.333333,151.847351,147.497314,155.843201,63.941635,63.360538,57.968090,...,0.061260,0.048903,0.054233,0.037589,0.057592,0.039946,0.019950,0.014866,0.011446,train
1,butterfly (1).jpg,butterfly,0,0.665625,165.210327,153.734741,122.301636,31.479156,34.430649,41.247486,...,0.224078,0.220957,0.109388,0.061717,0.224078,0.148780,0.224078,0.192939,0.095919,test
2,butterfly (1).png,butterfly,0,1.200750,31.207031,26.439392,28.153381,73.013542,57.023968,57.665653,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,train
3,butterfly (10).jpeg,butterfly,0,1.333333,55.270874,60.944275,34.758972,44.147003,46.827644,38.350193,...,0.260679,0.032350,0.117381,0.281991,0.281991,0.281991,0.099470,0.066883,0.073926,train
4,butterfly (10).jpg,butterfly,0,1.333333,110.409607,114.204834,79.344299,31.464451,31.279758,40.621906,...,0.168179,0.009444,0.011619,0.002050,0.030657,0.009936,0.291952,0.370398,0.124318,train


In [3]:
import pandas as pd

df = pd.read_csv("feature2.csv")

# ساخت ستون برچسب عددی از روی نام کلاس
df["label"] = df["class"].astype("category").cat.codes

target_column = "label"
if target_column not in df.columns:
    raise KeyError(f"Target column '{target_column}' not found in the dataset.")

y = df[target_column]
X_raw = df.drop(columns=[target_column])

non_numeric_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
if non_numeric_cols:
    logging.info("Dropping non-numeric columns: %s", non_numeric_cols)
else:
    logging.info("No non-numeric columns detected among features.")

X = df.select_dtypes(include=["int64", "float64"]).drop(columns=[target_column], errors="ignore")

if X.empty:
    raise ValueError("No numeric feature columns remain after dropping non-numeric columns.")

logging.info("Final feature matrix shape after cleaning: %s", X.shape)


2025-12-10 08:34:57,135 [INFO] Dropping non-numeric columns: ['filename', 'class', 'split']
2025-12-10 08:34:57,325 [INFO] Final feature matrix shape after cleaning: (26179, 1865)
2025-12-10 08:34:57,325 [INFO] Final feature matrix shape after cleaning: (26179, 1865)


## Section 2: Drop non-numeric columns
Remove all non-numeric feature columns (e.g., filenames/paths) before training.

In [4]:
target_column = "label"
if target_column not in df.columns:
    raise KeyError(f"Target column '{target_column}' not found in the dataset.")

y = df[target_column]
X_raw = df.drop(columns=[target_column])

non_numeric_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
if non_numeric_cols:
    logging.info("Dropping non-numeric columns: %s", non_numeric_cols)
else:
    logging.info("No non-numeric columns detected among features.")

X = df.select_dtypes(include=["int64", "float64"]).drop(columns=[target_column], errors="ignore")
X = X.astype(np.float32)

if X.empty:
    raise ValueError("No numeric feature columns remain after dropping non-numeric columns.")

logging.info("Final feature matrix shape after cleaning: %s", X.shape)

2025-12-10 08:35:01,635 [INFO] Dropping non-numeric columns: ['filename', 'class', 'split']
2025-12-10 08:35:01,943 [INFO] Final feature matrix shape after cleaning: (26179, 1865)
2025-12-10 08:35:01,943 [INFO] Final feature matrix shape after cleaning: (26179, 1865)


## Section 3: Train/Test split
Use stratified split so label proportions are preserved.

In [5]:

labels = list(pd.unique(y))
try:
    labels = sorted(labels)
except TypeError:
    labels = sorted(labels, key=lambda val: str(val))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logging.info("Train shape: %s; Test shape: %s", X_train.shape, X_test.shape)
logging.info("Train label distribution: %s", y_train.value_counts(normalize=True).round(3).to_dict())
logging.info("Test label distribution: %s", y_test.value_counts(normalize=True).round(3).to_dict())

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
logging.info("Using StratifiedKFold (n_splits=3, shuffle=True, random_state=42) for all model searches.")


2025-12-10 08:35:03,920 [INFO] Train shape: (20943, 1865); Test shape: (5236, 1865)
2025-12-10 08:35:03,922 [INFO] Train label distribution: {4: 0.186, 8: 0.184, 2: 0.118, 6: 0.1, 0: 0.081, 3: 0.071, 9: 0.071, 7: 0.07, 1: 0.064, 5: 0.055}
2025-12-10 08:35:03,923 [INFO] Test label distribution: {4: 0.186, 8: 0.184, 2: 0.118, 6: 0.1, 0: 0.081, 3: 0.071, 9: 0.071, 7: 0.07, 1: 0.064, 5: 0.055}
2025-12-10 08:35:03,924 [INFO] Using StratifiedKFold (n_splits=3, shuffle=True, random_state=42) for all model searches.
2025-12-10 08:35:03,922 [INFO] Train label distribution: {4: 0.186, 8: 0.184, 2: 0.118, 6: 0.1, 0: 0.081, 3: 0.071, 9: 0.071, 7: 0.07, 1: 0.064, 5: 0.055}
2025-12-10 08:35:03,923 [INFO] Test label distribution: {4: 0.186, 8: 0.184, 2: 0.118, 6: 0.1, 0: 0.081, 3: 0.071, 9: 0.071, 7: 0.07, 1: 0.064, 5: 0.055}
2025-12-10 08:35:03,924 [INFO] Using StratifiedKFold (n_splits=3, shuffle=True, random_state=42) for all model searches.


In [6]:
# Initialize all rows as empty
df['split'] = ''

# Mark train indices
df.loc[X_train.index, 'split'] = 'train'

# Mark test indices
df.loc[X_test.index, 'split'] = 'test'

# Save the updated dataframe back to CSV
output_csv = "feature2.csv"
df.to_csv(output_csv, index=False)

logging.info("Added 'split' column to dataframe and saved to %s", output_csv)
logging.info("Train samples: %d, Test samples: %d", 
             (df['split'] == 'train').sum(), 
             (df['split'] == 'test').sum())


2025-12-10 08:36:02,679 [INFO] Added 'split' column to dataframe and saved to feature2.csv
2025-12-10 08:36:02,683 [INFO] Train samples: 20943, Test samples: 5236
2025-12-10 08:36:02,683 [INFO] Train samples: 20943, Test samples: 5236


In [6]:
# # Save train/test split images to Dataset/split-animal/<class>/train and /test
# # This organizes images into separate folders based on their class and split
# import os
# import shutil
# from pathlib import Path

# logging.info("Starting to organize images into train/test folders...")

# out_base = Path("Dataset") / "split-animal"
# out_base.mkdir(parents=True, exist_ok=True)

# # Create directory structure for each class
# classes = pd.unique(df['class'])
# for cls in classes:
#     (out_base / cls / "train").mkdir(parents=True, exist_ok=True)
#     (out_base / cls / "test").mkdir(parents=True, exist_ok=True)

# copied = 0
# missing = 0

# # Base path for source images
# source_base = Path("Dataset") / "Animals-10"

# # Process each row in the dataframe
# for idx, row in df.iterrows():
#     # Determine if this sample is train or test
#     split_type = row['split']
#     if split_type not in ['train', 'test']:
#         logging.warning("Row %d has invalid split value: %s - skipping", idx, split_type)
#         continue
    
#     # Construct source path: Dataset/Animals-10/<class>/<filename>
#     class_name = row['class']
#     filename = row['filename']
#     candidate = source_base / class_name / filename
    
#     if not candidate.exists():
#         logging.warning("Source file not found: %s (index=%s)", candidate, idx)
#         missing += 1
#         continue
    
#     # Destination path: Dataset/split-animal/<class>/<train|test>/<filename>
#     dest = out_base / class_name / split_type / filename
    
#     try:
#         shutil.copy2(candidate, dest)
#         copied += 1
#         if copied % 500 == 0:  # Progress indicator every 500 files
#             logging.info("Progress: %d files copied so far...", copied)
#     except Exception as e:
#         logging.warning("Failed to copy %s -> %s : %s", candidate, dest, e)
#         missing += 1

# logging.info("Image organization complete: %d files copied, %d errors/missing", copied, missing)
# logging.info("Output directory: %s", out_base.resolve())
# logging.info("Train images: %d, Test images: %d", 
#              (df['split'] == 'train').sum(), 
#              (df['split'] == 'test').sum())


## Section 4: SVM training + tuning + evaluation

In [38]:
# import logging
# import pandas as pd
# from sklearn.svm import SVC
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# # فرض: X_train, X_test, y_train, y_test, labels قبلاً تعریف شده‌اند

# # -------------------------------
# # 1) نرمال‌سازی فیچرها
# # -------------------------------
# logging.info("Scaling features for baseline SVM with StandardScaler...")

# scaler_baseline = StandardScaler()
# X_train_scaled = scaler_baseline.fit_transform(X_train)
# X_test_scaled = scaler_baseline.transform(X_test)

# # -------------------------------
# # 2) تنظیم دستی C و gamma
# #    این مقادیر را هر طور خواستی عوض کن
# # -------------------------------
# C_value = 5       # مثلا 10
# gamma_value = 0.1   # مثلا 0.01

# logging.info(
#     "Training baseline SVM with manual hyperparameters: C=%.4f, gamma=%.4f, kernel='rbf'",
#     C_value,
#     gamma_value,
# )

# svm_baseline = SVC(
#     C=C_value,
#     gamma=gamma_value,
#     kernel="rbf",
#     random_state=42
# )

# # -------------------------------
# # 3) آموزش مدل روی داده‌های نرمال‌شده
# # -------------------------------
# svm_baseline.fit(X_train_scaled, y_train)

# # -------------------------------
# # 4) پیش‌بینی و دقت روی Train و Test
# # -------------------------------
# y_train_pred = svm_baseline.predict(X_train_scaled)
# y_test_pred = svm_baseline.predict(X_test_scaled)

# train_acc = accuracy_score(y_train, y_train_pred)
# test_acc = accuracy_score(y_test, y_test_pred)

# logging.info("Baseline SVM (manual C,gamma) TRAIN accuracy: %.4f", train_acc)
# logging.info("Baseline SVM (manual C,gamma) TEST  accuracy: %.4f", test_acc)

# # -------------------------------
# # 5) گزارش و ماتریس سردرگمی
# # -------------------------------
# svm_baseline_report = classification_report(y_test, y_test_pred)
# logging.info("Baseline SVM classification report:\n%s", svm_baseline_report)

# svm_baseline_cm = confusion_matrix(y_test, y_test_pred, labels=labels)
# svm_baseline_cm_df = pd.DataFrame(
#     svm_baseline_cm,
#     index=[f"true_{lbl}" for lbl in labels],
#     columns=[f"pred_{lbl}" for lbl in labels]
# )
# logging.info("Baseline SVM confusion matrix:\n%s", svm_baseline_cm_df)


In [7]:
import logging
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# فرض: X_train, X_test, y_train, y_test, labels قبلاً تعریف شده‌اند


logging.info("Scaling features for manual SVM check with StandardScaler...")

scaler_manual = StandardScaler()
X_train_scaled = scaler_manual.fit_transform(X_train)
X_test_scaled = scaler_manual.transform(X_test)

# -------------------------------
# 2) تنظیم دستی پارامترها: C=10, gamma='scale', kernel='rbf'
# -------------------------------
C_value = 10
gamma_value = 'scale'
kernel_value = 'rbf'

logging.info(
    "Training manual SVM with hyperparameters: C=%.4f, gamma=%s, kernel=%s",
    C_value,
    gamma_value,
    kernel_value,
 )

svm_manual = SVC(
    C=C_value,
    gamma=gamma_value,
    kernel=kernel_value,
    random_state=42
 )
svm_manual.fit(X_train_scaled, y_train)


y_train_pred = svm_manual.predict(X_train_scaled)
y_test_pred = svm_manual.predict(X_test_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

logging.info("Manual SVM TRAIN accuracy: %.4f", train_acc)
logging.info("Manual SVM TEST  accuracy: %.4f", test_acc)


svm_manual_report = classification_report(y_test, y_test_pred)
logging.info("Manual SVM classification report:\n%s", svm_manual_report)

svm_manual_cm = confusion_matrix(y_test, y_test_pred, labels=labels)
svm_manual_cm_df = pd.DataFrame(
    svm_manual_cm,
    index=[f"true_{lbl}" for lbl in labels],
    columns=[f"pred_{lbl}" for lbl in labels]
 )
logging.info("Manual SVM confusion matrix:\n%s", svm_manual_cm_df)

joblib.dump({"model": svm_manual, "scaler": scaler_manual}, "svm_manual_scaled.pkl")
logging.info("Saved manual SVM and scaler to svm_manual_scaled.pkl")

2025-12-10 08:36:16,124 [INFO] Scaling features for manual SVM check with StandardScaler...
2025-12-10 08:36:16,682 [INFO] Training manual SVM with hyperparameters: C=10.0000, gamma=scale, kernel=rbf
2025-12-10 08:36:16,682 [INFO] Training manual SVM with hyperparameters: C=10.0000, gamma=scale, kernel=rbf
2025-12-10 09:03:40,777 [INFO] Manual SVM TRAIN accuracy: 1.0000
2025-12-10 09:03:40,778 [INFO] Manual SVM TEST  accuracy: 0.6492
2025-12-10 09:03:40,790 [INFO] Manual SVM classification report:
              precision    recall  f1-score   support

           0       0.72      0.67      0.69       422
           1       0.58      0.35      0.44       334
           2       0.66      0.70      0.67       620
           3       0.64      0.58      0.61       373
           4       0.54      0.72      0.62       973
           5       0.69      0.50      0.58       289
           6       0.72      0.68      0.70       525
           7       0.57      0.51      0.54       364
          

In [15]:
import time
from scipy.stats import loguniform
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

logging.info("Starting SVM tuning with HalvingRandomSearchCV + PCA for faster convergence...")

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, whiten=True, random_state=42)),
    ("svm", SVC(random_state=42)),
])

svm_param_distributions = {
    "pca__n_components": [64, 96, 128, 160, 192, 224],
    "svm__C": loguniform(1e-1, 1e2),
    "svm__gamma": loguniform(1e-4, 1e-1),
    "svm__kernel": ["rbf"],
}

svm_search = HalvingRandomSearchCV(
    estimator=svm_pipeline,
    param_distributions=svm_param_distributions,
    factor=3,
    resource="n_samples",
    max_resources="auto",
    min_resources=600,
    aggressive_elimination=True,
    n_candidates=30,
    scoring="accuracy",
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=False,
    refit=True,
 )

start = time.perf_counter()
svm_search.fit(X_train, y_train)
elapsed_min = (time.perf_counter() - start) / 60.0
logging.info("SVM HalvingRandomSearchCV complete (%.2f minutes).", elapsed_min)

logging.info("Halving stages - resources: %s", svm_search.n_resources_)
logging.info("Halving stages - candidates: %s", svm_search.n_candidates_)

logging.info("Best SVM parameters found: %s", svm_search.best_params_)
logging.info(
    "Best cross-validated accuracy (mean validation accuracy over folds): %.4f",
    svm_search.best_score_,
)

best_svm = svm_search.best_estimator_

y_train_pred = best_svm.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
logging.info("Train accuracy: %.4f", train_acc)

y_test_pred = best_svm.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)
logging.info("Test accuracy: %.4f", test_acc)

svm_report = classification_report(y_test, y_test_pred)
svm_cm = confusion_matrix(y_test, y_test_pred)

logging.info("Tuned SVM classification report:\n%s", svm_report)
logging.info("Tuned SVM confusion matrix:\n%s", svm_cm)

joblib.dump(best_svm, "svm_tuned_halving_pca.pkl")
logging.info("Saved tuned SVM model to svm_tuned_halving_pca.pkl")

2025-12-10 08:18:09,536 [INFO] Starting SVM tuning with HalvingRandomSearchCV + PCA for faster convergence...


n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 4
min_resources_: 600
max_resources_: 20943
aggressive_elimination: True
factor: 3
----------
iter: 0
n_candidates: 30
n_resources: 600
Fitting 3 folds for each of 30 candidates, totalling 90 fits
----------
iter: 1
n_candidates: 10
n_resources: 1800
Fitting 3 folds for each of 10 candidates, totalling 30 fits
----------
iter: 1
n_candidates: 10
n_resources: 1800
Fitting 3 folds for each of 10 candidates, totalling 30 fits
----------
iter: 2
n_candidates: 4
n_resources: 5400
Fitting 3 folds for each of 4 candidates, totalling 12 fits
----------
iter: 2
n_candidates: 4
n_resources: 5400
Fitting 3 folds for each of 4 candidates, totalling 12 fits
----------
iter: 3
n_candidates: 2
n_resources: 16200
Fitting 3 folds for each of 2 candidates, totalling 6 fits
----------
iter: 3
n_candidates: 2
n_resources: 16200
Fitting 3 folds for each of 2 candidates, totalling 6 fits


2025-12-10 08:20:52,904 [INFO] SVM HalvingRandomSearchCV complete (2.72 minutes).
2025-12-10 08:20:52,906 [INFO] Halving stages - resources: [600, 1800, 5400, 16200]
2025-12-10 08:20:52,907 [INFO] Halving stages - candidates: [30, 10, 4, 2]
2025-12-10 08:20:52,908 [INFO] Best SVM parameters found: {'pca__n_components': 192, 'svm__C': np.float64(6.17377039470457), 'svm__gamma': np.float64(0.002175195311877766), 'svm__kernel': 'rbf'}
2025-12-10 08:20:52,908 [INFO] Best cross-validated accuracy (mean validation accuracy over folds): 0.5499
2025-12-10 08:20:52,906 [INFO] Halving stages - resources: [600, 1800, 5400, 16200]
2025-12-10 08:20:52,907 [INFO] Halving stages - candidates: [30, 10, 4, 2]
2025-12-10 08:20:52,908 [INFO] Best SVM parameters found: {'pca__n_components': 192, 'svm__C': np.float64(6.17377039470457), 'svm__gamma': np.float64(0.002175195311877766), 'svm__kernel': 'rbf'}
2025-12-10 08:20:52,908 [INFO] Best cross-validated accuracy (mean validation accuracy over folds): 0.5

## Section 5: Decision Tree training + tuning + evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score  # اگر قبلاً ایمپورت نکردی

dt_param_distributions = {
    "max_depth": [5, 10, 20, None],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 5, 10],
}

dt_n_iter = 16
logging.info(
    "Running Decision Tree RandomizedSearchCV (n_iter=%d, random_state=42) over %d candidate values.",
    dt_n_iter,
    len(dt_param_distributions["max_depth"])
    * len(dt_param_distributions["criterion"])
    * len(dt_param_distributions["min_samples_split"]),
)

dt_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=dt_param_distributions,
    n_iter=dt_n_iter,
    random_state=42,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
logging.info("Fitting Decision Tree randomized search...")
dt_search.fit(X_train, y_train)
logging.info("Decision Tree RandomizedSearchCV complete.")

best_dt = dt_search.best_estimator_
logging.info("Best Decision Tree parameters: %s", dt_search.best_params_)
logging.info("Best cross-validated accuracy: %.4f", dt_search.best_score_)

# -------------------------------
# دقت روی TRAIN و TEST برای مدل تیون‌شده
# -------------------------------
dt_train_pred = best_dt.predict(X_train)
dt_test_pred = best_dt.predict(X_test)

dt_train_acc = accuracy_score(y_train, dt_train_pred)
dt_test_acc = accuracy_score(y_test, dt_test_pred)

logging.info("Decision Tree TRAIN accuracy: %.4f", dt_train_acc)
logging.info("Decision Tree TEST  accuracy: %.4f", dt_test_acc)

# -------------------------------
# گزارش و ماتریس سردرگمی روی داده‌های تست
# -------------------------------
dt_tuned_report = classification_report(y_test, dt_test_pred)
logging.info("Decision Tree classification report:\n%s", dt_tuned_report)

dt_tuned_cm = confusion_matrix(y_test, dt_test_pred, labels=labels)
dt_tuned_cm_df = pd.DataFrame(
    dt_tuned_cm,
    index=[f"true_{lbl}" for lbl in labels],
    columns=[f"pred_{lbl}" for lbl in labels]
)
logging.info("Decision Tree confusion matrix:\n%s", dt_tuned_cm_df)


2025-12-09 16:38:49,786 - INFO - Running Decision Tree RandomizedSearchCV (n_iter=16, random_state=42) over 24 candidate values.
2025-12-09 16:38:49,787 - INFO - Fitting Decision Tree randomized search...
2025-12-09 16:38:49,787 - INFO - Fitting Decision Tree randomized search...


Fitting 5 folds for each of 16 candidates, totalling 80 fits


2025-12-09 16:39:22,873 - INFO - Decision Tree RandomizedSearchCV complete.
2025-12-09 16:39:22,874 - INFO - Best Decision Tree parameters: {'min_samples_split': 2, 'max_depth': 10, 'criterion': 'gini'}
2025-12-09 16:39:22,874 - INFO - Best cross-validated accuracy: 0.3610
2025-12-09 16:39:22,885 - INFO - Decision Tree TRAIN accuracy: 0.4895
2025-12-09 16:39:22,874 - INFO - Best Decision Tree parameters: {'min_samples_split': 2, 'max_depth': 10, 'criterion': 'gini'}
2025-12-09 16:39:22,874 - INFO - Best cross-validated accuracy: 0.3610
2025-12-09 16:39:22,885 - INFO - Decision Tree TRAIN accuracy: 0.4895
2025-12-09 16:39:22,886 - INFO - Decision Tree TEST  accuracy: 0.3678
2025-12-09 16:39:22,894 - INFO - Decision Tree classification report:
              precision    recall  f1-score   support

           0       0.60      0.52      0.56       422
           1       0.23      0.13      0.17       334
           2       0.37      0.48      0.41       620
           3       0.26      0.

## Section 6: Save models
Persist tuned models for reuse.

In [ ]:

svm_model_path = Path("svm_phase3.pkl")
dt_model_path = Path("dt_phase3.pkl")

logging.info("Saving tuned models to disk...")
joblib.dump(best_svm, svm_model_path)
joblib.dump(best_dt, dt_model_path)

logging.info("Saved tuned SVM model to: %s", svm_model_path.resolve())
logging.info("Saved tuned Decision Tree model to: %s", dt_model_path.resolve())


2025-12-09 16:39:22,907 - INFO - Saving tuned models to disk...
2025-12-09 16:39:22,950 - INFO - Saved tuned SVM model to: E:\7\computational Intelligence\projects\project2\project2\svm_phase3.pkl
2025-12-09 16:39:22,952 - INFO - Saved tuned Decision Tree model to: E:\7\computational Intelligence\projects\project2\project2\dt_phase3.pkl
2025-12-09 16:39:22,950 - INFO - Saved tuned SVM model to: E:\7\computational Intelligence\projects\project2\project2\svm_phase3.pkl
2025-12-09 16:39:22,952 - INFO - Saved tuned Decision Tree model to: E:\7\computational Intelligence\projects\project2\project2\dt_phase3.pkl
